# Release metro waves

Computes release waves via topological sort of inter-repository dependencies. Wave 0 contains repositories with no internal dependencies that can release first. Each subsequent wave depends only on earlier waves.

In [ ]:
repository_filter: list[str] = []
data_file_dependencies: str = "../samples/release_metro_dependencies_in_use.csv"
data_file_parents: str = "../samples/release_metro_parent_relationships.csv"

In [ ]:
from code_data_science import (
    data_table as dt,
    unique_dictionaries as ud,
    tree_data_grid,
)
import pandas as pd
from moderne_visualizations_misc.reusable.release_metro_utils import (
    build_release_graph,
    compute_release_waves,
)
from moderne_visualizations_misc.reusable.quality_utils import (
    filter_repos,
    short_repo,
)
import warnings

warnings.simplefilter("ignore")

coords_df = dt.read_csv("../samples/project_coordinates.csv")
deps_df = pd.read_csv(data_file_dependencies, on_bad_lines="skip")
# ParentRelationships may be empty (no headers) when no parent POMs are found.
try:
    parents_df = pd.read_csv(data_file_parents, on_bad_lines="skip")
except pd.errors.EmptyDataError:
    parents_df = pd.DataFrame()

coords_df = filter_repos(coords_df, repository_filter)
deps_df = filter_repos(deps_df, repository_filter)
if len(parents_df) > 0:
    parents_df = filter_repos(parents_df, repository_filter)

In [ ]:
G = build_release_graph(coords_df, deps_df, parents_df)
waves, circular = compute_release_waves(G)

total_modules = len(G.nodes())
releasable = sum(len(w) for w in waves)
wave_count = len(waves)

tree = ud.UniqueDictionaries()

for i, wave in enumerate(waves):
    wave_label = f"Wave {i} ({len(wave)} modules)"
    for repo in sorted(wave):
        tree.add({
            "path": f"Release Waves/{wave_label}/{short_repo(repo)}",
            "depends_on": str(G.in_degree(repo)),
            "required_by": str(G.out_degree(repo)),
        })

if circular:
    circ_label = f"Circular ({len(circular)} modules)"
    for repo in circular:
        tree.add({
            "path": f"Release Waves/{circ_label}/{short_repo(repo)}",
            "depends_on": str(G.in_degree(repo)),
            "required_by": str(G.out_degree(repo)),
        })

tree_data_grid.display(
    tree.to_list(),
    f"Release Waves — {releasable} of {total_modules} modules in {wave_count} waves",
    [
        {"field": "depends_on", "headerName": "Depends on", "minWidth": 120},
        {"field": "required_by", "headerName": "Required by", "minWidth": 120},
    ],
)